## Random sampling

Uniform draws over the two things that vary: how wide the doorway is, and when the person
is in the way. It assumes nothing and covers everything badly, which is the point — it is
the reference every other strategy here is read against.

**What to look for:** the dots land anywhere, including clumps and gaps. A failure region
narrower than the gaps between them is one this campaign can miss entirely.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json, os, sqlite3
import pandas as pd
import matplotlib.pyplot as plt

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from campaign.db rather than the results index because that is where a SEARCH records
    what it scored -- the index holds per-run tables, and a search's unit of analysis is the cell.
    """
    db = os.path.join(data_dir, 'campaign.db')
    if not os.path.exists(db):
        return pd.DataFrame()
    with sqlite3.connect(db) as conn:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'Random — the honest denominator'

# Where it sampled, and what it found there. Two axes because the space has two -- no
# projection, no facets, nothing to reconstruct in your head.
if not scored.empty:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    pts = ax.scatter(scored['gap_width'], scored['walker_dwell'],
                     c=scored['robustness'], cmap='RdYlGn', vmin=-1, vmax=1,
                     s=90, edgecolor='black', linewidth=0.4)
    fig.colorbar(pts, ax=ax, label='robustness  (< 0 failed)')
    ax.set_xlabel('doorway width [m]')
    ax.set_ylabel("walker dwell [s]  (when it is in the way)")
    ax.set_title('%s: %d cells' % (TITLE, len(scored)))
    plt.tight_layout(); plt.show()


In [ ]:
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst robustness      : {scored['robustness'].min():.3f}")
    print()
    import math
    n = len(scored); k = int(failed); phat = k / n
    z = 1.96
    denom = 1 + z * z / n
    centre = (phat + z * z / (2 * n)) / denom
    half = z * math.sqrt(phat * (1 - phat) / n + z * z / (4 * n * n)) / denom
    print("What this campaign is FOR -- the fraction of the space that fails:")
    print(f"  {phat:.1%} of cells  (95% Wilson interval {centre - half:.1%} .. {centre + half:.1%})")
    print(f"  interval width      : {2 * half:.1%} over {n} cells")
    print("  A DIFFERENT strategy's fraction is read against this one. To compare the")
    print("  WIDTH of two intervals, run each over several seed: values -- a spread is a")
    print("  property of an estimator across repeats, not something one campaign reports.")
